In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, classification_report, confusion_matrix, roc_curve, auc
import tensorflow as tf
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import Input, Dense, Dropout, Conv1D, MaxPooling1D, Flatten, LSTM, TimeDistributed, BatchNormalization, GlobalAveragePooling1D, Reshape, RepeatVector
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
import joblib
import pickle
import warnings
warnings.filterwarnings("ignore")
tf.random.set_seed(42)
np.random.seed(42)


In [1]:
%%writefile streamlit_app.py

import streamlit as st
import pandas as pd
import numpy as np
import tensorflow as tf
import joblib
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, classification_report, confusion_matrix, roc_curve

# --- Configuration --- #
MODEL_DIR = "/content/Friday-WorkingHours-Morning.pcap_ISCX.csv"
SEQUENCE_LENGTH = 10
LABEL_COL = 'label'

# --- Load Models and Preprocessors ---
@st.cache_resource
def load_artifacts():
    try:
        scaler = joblib.load(os.path.join(MODEL_DIR, "scaler.pkl"))
        le = joblib.load(os.path.join(MODEL_DIR, "label_encoder.pkl"))
        clf = tf.keras.models.load_model(os.path.join(MODEL_DIR, 'cnn_lstm_best.h5'))
        ae = tf.keras.models.load_model(os.path.join(MODEL_DIR, 'lstm_ae_best.h5'))
        # Load the anomaly threshold
        mse_test = np.load(os.path.join(MODEL_DIR, 'mse_test.npy'))
        ae_threshold = np.percentile(mse_test[np.where(np.load(os.path.join(MODEL_DIR, 'y_test.npy')) == 0)[0]], 95)
        return scaler, le, clf, ae, ae_threshold
    except Exception as e:
        st.error(f"Error loading artifacts: {e}. Make sure the models and preprocessors are trained and saved in {MODEL_DIR}.")
        st.stop()

scaler, le, clf, ae, ae_threshold = load_artifacts()

# --- Data Preprocessing Function (similar to notebook) ---
def preprocess_data(df_raw):
    df = df_raw.copy()

    # Standardize column names
    df.columns = df.columns.str.strip().str.lower()

    # Drop identified non-numeric/high cardinality columns during training, if present
    # (assuming this list is consistent with training or handled robustly)
    drop_cols_identified_during_training = [] # Add specific columns if they were dropped during notebook execution
    for c in df.columns:
        if df[c].dtype == object and c not in ['label','attack','class','flow_id']:
            if df[c].nunique() > 1000:
                drop_cols_identified_during_training.append(c)

    if drop_cols_identified_during_training:
        df = df.drop(columns=[col for col in drop_cols_identified_during_training if col in df.columns])

    # Convert label column to string and strip whitespace if it exists
    if LABEL_COL in df.columns:
        df[LABEL_COL] = df[LABEL_COL].astype(str).str.strip()
        # Store raw labels for comparison if needed
        y_raw_pred = df[LABEL_COL]
        # We only care about the features for prediction, actual labels are for evaluation.
        # If this is unseen data, we don't have true labels.
        # For consistent sequence creation, we will create a dummy y if no label is present.
        # If label column is present, we'll try to encode it for the sequence function.
        try:
            y_enc_dummy = le.transform(df[LABEL_COL].apply(lambda v: 'Benign' if v.lower() in ['benign','normal','normal traffic'] or 'benign' in v.lower() else 'Attack'))
        except:
            y_enc_dummy = np.zeros(len(df)) # Dummy labels if new data has unknown labels
    else:
        y_raw_pred = pd.Series(['Unknown'] * len(df))
        y_enc_dummy = np.zeros(len(df))


    # Keep only numeric features for modeling
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if not numeric_cols:
        st.error("No numeric features found in the uploaded data after initial processing. Please check your dataset.")
        st.stop()
    X_num = df[numeric_cols]

    # Handle infinite values and NaNs
    X_num = X_num.replace([np.inf, -np.inf], np.nan)
    X_num = X_num.fillna(0)

    # Scale features
    X_scaled = scaler.transform(X_num)

    # Create sequences (modified for prediction without true labels)
    # The make_sequences function needs to be adapted slightly for inference
    def make_sequences_inference(X, seq_len=SEQUENCE_LENGTH, stride=1):
        Xs = []
        for start in range(0, X.shape[0] - seq_len + 1, stride):
            end = start + seq_len
            Xs.append(X[start:end])
        return np.array(Xs)

    X_seq_pred = make_sequences_inference(X_scaled, seq_len=SEQUENCE_LENGTH)

    # We need to ensure that the number of features matches the training data
    if X_seq_pred.shape[2] != clf.input_shape[2]:
        st.error(f"The number of features in the uploaded data ({X_seq_pred.shape[2]}) does not match the model's expected input features ({clf.input_shape[2]}). Please ensure your dataset has the same columns as the training data.")
        st.stop()

    return X_seq_pred, X_num.columns.tolist() # Also return feature names


# --- Prediction Functions ---
def predict_classifier(X_seq):
    y_pred_prob = clf.predict(X_seq).ravel()
    y_pred_class = (y_pred_prob >= 0.5).astype(int)
    return y_pred_class, y_pred_prob

def predict_autoencoder(X_seq):
    recon = ae.predict(X_seq)
    mse = np.mean(np.mean(np.square(recon - X_seq), axis=2), axis=1)
    y_pred_ae = (mse > ae_threshold).astype(int)
    return y_pred_ae, mse

# --- Streamlit UI --- #
st.set_page_config(page_title="IDS with CNN-LSTM & LSTM-AE", layout="wide")

st.markdown("<h1 style='text-align: center; color: #2E8B57;'>Intrusion Detection System</h1>", unsafe_allow_html=True)
st.markdown("<h3 style='text-align: center; color: #3CB371;'>Detecting Network Intrusions using Deep Learning</h3>", unsafe_allow_html=True)

st.write("Upload a CSV file containing network traffic data to analyze it for intrusions.")

uploaded_file = st.file_uploader("Choose a CSV file", type="csv")

if uploaded_file is not None:
    df_upload = pd.read_csv(uploaded_file)
    st.subheader("Uploaded Data Preview")
    st.write(df_upload.head())

    with st.spinner("Preprocessing data and making predictions..."):
        try:
            X_seq_new, feature_names = preprocess_data(df_upload)

            if len(X_seq_new) == 0:
                st.warning(f"Not enough data points to form sequences of length {SEQUENCE_LENGTH}. Please upload more data.")
            else:
                # --- CNN-LSTM Classifier Prediction ---
                st.markdown("<hr style='border: 1px solid #ddd;'/>", unsafe_allow_html=True)
                st.markdown("<h2 style='color: #2F4F4F;'>1. CNN-LSTM Classifier Results</h2>", unsafe_allow_html=True)

                y_pred_clf, y_pred_prob_clf = predict_classifier(X_seq_new)

                # Get labels from the loaded LabelEncoder
                attack_label_idx = np.where(le.classes_ == 'Attack')[0]
                benign_label_idx = np.where(le.classes_ == 'Benign')[0]

                # Map predictions to original labels (Attack/Benign)
                predicted_labels_clf = np.array([le.classes_[label] for label in y_pred_clf])

                st.write(f"Detected {np.sum(predicted_labels_clf == 'Attack')} potential attack sequences out of {len(predicted_labels_clf)}.")

                col1, col2 = st.columns(2)
                with col1:
                    fig_clf_dist = plt.figure(figsize=(6, 4))
                    sns.countplot(x=predicted_labels_clf, palette='viridis')
                    plt.title('CNN-LSTM Prediction Distribution')
                    st.pyplot(fig_clf_dist)
                with col2:
                    st.write("### Prediction Breakdown (Classifier)")
                    clf_results_df = pd.DataFrame({
                        'Prediction': predicted_labels_clf,
                        'Confidence (Attack)': y_pred_prob_clf
                    })
                    st.dataframe(clf_results_df.head(10))
                    if np.sum(predicted_labels_clf == 'Attack') > 0:
                        st.markdown(f"<p style='color: #FF4500;'>**Warning:** The CNN-LSTM model has detected anomalies based on its training patterns.</p>", unsafe_allow_html=True)

                # --- LSTM Autoencoder Anomaly Detection ---
                st.markdown("<hr style='border: 1px solid #ddd;'/>", unsafe_allow_html=True)
                st.markdown("<h2 style='color: #2F4F4F;'>2. LSTM Autoencoder Anomaly Detection</h2>", unsafe_allow_html=True)

                y_pred_ae, mse_values = predict_autoencoder(X_seq_new)
                predicted_labels_ae = np.array([le.classes_[label] for label in y_pred_ae])

                st.write(f"Detected {np.sum(predicted_labels_ae == 'Attack')} potential attack sequences out of {len(predicted_labels_ae)} (using reconstruction error threshold).")
                st.write(f"Anomaly Threshold (MSE): **{ae_threshold:.4f}**")

                col3, col4 = st.columns(2)
                with col3:
                    fig_ae_dist = plt.figure(figsize=(6, 4))
                    sns.histplot(mse_values, bins=50, kde=True, color='red')
                    plt.axvline(ae_threshold, color='blue', linestyle='--', label=f'Threshold: {ae_threshold:.2f}')
                    plt.title('Reconstruction Error (MSE) Distribution')
                    plt.xlabel('Mean Squared Error (MSE)')
                    plt.ylabel('Frequency')
                    plt.legend()
                    st.pyplot(fig_ae_dist)
                with col4:
                    st.write("### Anomaly Detection Breakdown (Autoencoder)")
                    ae_results_df = pd.DataFrame({
                        'Prediction': predicted_labels_ae,
                        'Reconstruction Error (MSE)': mse_values
                    })
                    st.dataframe(ae_results_df.head(10))
                    if np.sum(predicted_labels_ae == 'Attack') > 0:
                        st.markdown(f"<p style='color: #FF4500;'>**Warning:** The LSTM Autoencoder model has detected anomalies based on high reconstruction error.</p>", unsafe_allow_html=True)


        except Exception as e:
            st.error(f"An error occurred during prediction: {e}")
            st.info("Please ensure the uploaded CSV has similar structure and features as the training data.")

else:
    st.info("Please upload a CSV file to begin the intrusion detection analysis.")

st.sidebar.markdown("""
<div style='text-align: center;'>
    <h2>About This App</h2>
    <p>This application demonstrates an Intrusion Detection System (IDS) using two deep learning models:</p>
    <ul>
        <li><b>CNN-LSTM Classifier:</b> A supervised model trained to classify network traffic sequences as 'Benign' or 'Attack'.</li>
        <li><b>LSTM Autoencoder:</b> An unsupervised anomaly detection model trained on benign traffic to identify deviations (potential attacks) based on reconstruction error.</li>
    </ul>
    <p>Upload your network traffic data in CSV format, and the system will provide predictions from both models.</p>
</div>
""", unsafe_allow_html=True)


Writing streamlit_app.py


In [ ]:
DATA_PATH = "/content/Friday-WorkingHours-Morning.pcap_ISCX.csv"  # change to your dataset
SAMPLE_FRACTION = 1.0  # set <1.0 to sample the CSV for faster tests
SEQUENCE_LENGTH = 10   # number of consecutive rows in a sequence window
TEST_SIZE = 0.2
RANDOM_STATE = 42
BATCH_SIZE = 256
EPOCHS = 40
AUTOENCODER_EPOCHS = 30
MODEL_DIR = "/content/models_ids"
os.makedirs(MODEL_DIR, exist_ok=True)

In [ ]:
print("Loading", DATA_PATH)
df = pd.read_csv(DATA_PATH)
print("Original shape:", df.shape)
if SAMPLE_FRACTION < 1.0:
    df = df.sample(frac=SAMPLE_FRACTION, random_state=RANDOM_STATE).reset_index(drop=True)
print("Using shape:", df.shape)

Loading /content/Friday-WorkingHours-Morning.pcap_ISCX.csv
Original shape: (191033, 79)
Using shape: (191033, 79)


In [ ]:
drop_cols = []
for c in df.columns:
    if df[c].dtype == object and c.lower() not in ['label','attack','class','flow_id']:
        # if a string column has many unique values, drop it by default
        if df[c].nunique() > 1000:
            drop_cols.append(c)

In [ ]:
drop_cols = []

# --- NEW CODE START ---
# Standardize column names by stripping whitespace and converting to lowercase
df.columns = df.columns.str.strip().str.lower()
# --- NEW CODE END ---

for c in df.columns:
    # if a string column has many unique values, drop it by default
    # 'c' is already lowercased and stripped now
    if df[c].dtype == object and c not in ['label','attack','class','flow_id']:
        if df[c].nunique() > 1000:
            drop_cols.append(c)

# keep label columns if present
# 'c' is already lowercased and stripped now
possible_label_cols = [c for c in df.columns if c in ('label','attack','class','flowlabel')]
print("Possible label columns:", possible_label_cols)

if len(possible_label_cols) > 0:
    LABEL_COL = possible_label_cols[0]
else:
    # If no label col found, try 'label' as all columns are now lowercased
    if 'label' in df.columns:
        LABEL_COL = 'label'
    else:
        raise ValueError("No label column found automatically. Rename your label column to 'Label' or 'label'.")

print("Using label column:", LABEL_COL)

# drop the chosen drop_cols
if drop_cols:
    print("Dropping columns:\n", drop_cols)
    df = df.drop(columns=drop_cols)

# Convert label column to string and strip whitespace
df[LABEL_COL] = df[LABEL_COL].astype(str).str.strip()

# Keep only numeric features for modeling
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print("Numeric features count:", len(numeric_cols))
if len(numeric_cols) == 0:
    raise ValueError("No numeric features available after filtering. Ensure your CSV has numeric features.")

X_num = df[numeric_cols]

# --- FIX START: Handle infinite values before filling NaNs and scaling ---
X_num = X_num.replace([np.inf, -np.inf], np.nan) # Replace inf with NaN
X_num = X_num.fillna(0) # Fill NaNs (original NaNs + new NaNs from inf) with 0
# --- FIX END ---

y_raw = df[LABEL_COL]

# Optionally map labels: combine many attack classes into 'Attack' vs 'Benign' or keep multi-class
BINARY_LABEL = True  # set False to keep multi-class
if BINARY_LABEL:
    y = y_raw.apply(lambda v: 'Benign' if v.lower() in ['benign','normal','normal traffic'] or 'benign' in v.lower() else 'Attack')
else:
    y = y_raw

le = LabelEncoder()
y_enc = le.fit_transform(y)
print("Label classes:", list(le.classes_))
# Save label encoder for later
joblib.dump(le, os.path.join(MODEL_DIR, "label_encoder.pkl"))

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_num)
joblib.dump(scaler, os.path.join(MODEL_DIR, "scaler.pkl"))

Possible label columns: ['label']
Using label column: label
Numeric features count: 78
Label classes: ['Attack', 'Benign']


['/content/models_ids/scaler.pkl']

In [ ]:
# Cell 4: create sliding-window sequences for sequence models (CNN+LSTM, LSTM-AE)
def make_sequences(X, y, seq_len=SEQUENCE_LENGTH, stride=1):
    Xs, ys = [], []
    for start in range(0, X.shape[0] - seq_len + 1, stride):
        end = start + seq_len
        Xs.append(X[start:end])
        # label a window as Attack if any row in the window is Attack (for supervised)
        ys.append(1 if np.any(y[start:end] != le.transform(['Benign'])[0]) else 0)
    return np.array(Xs), np.array(ys)

X_seq, y_seq = make_sequences(X_scaled, y_enc, seq_len=SEQUENCE_LENGTH, stride=1)
print("Sequences shape:", X_seq.shape, y_seq.shape)
# For classifier we need per-window labels y_seq. For autoencoder we'll train on benign windows only.

# Split into train/test
X_train, X_test, y_train, y_test = train_test_split(X_seq, y_seq, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y_seq)
print("Train windows:", X_train.shape, "Test windows:", X_test.shape)

Sequences shape: (191024, 10, 78) (191024,)
Train windows: (152819, 10, 78) Test windows: (38205, 10, 78)


In [ ]:
def build_cnn_lstm_classifier(timesteps, n_features):
    inp = Input(shape=(timesteps, n_features))
    # Conv block 1
    x = Conv1D(filters=64, kernel_size=3, padding='same', activation='relu')(inp)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2)(x)
    x = Dropout(0.2)(x)
    # Conv block 2
    x = Conv1D(filters=128, kernel_size=3, padding='same', activation='relu')(x)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2)(x)
    x = Dropout(0.2)(x)
    # LSTM on the reduced sequence
    x = LSTM(128, return_sequences=False)(x)
    x = Dropout(0.3)(x)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.3)(x)
    out = Dense(1, activation='sigmoid')(x)
    model = Model(inp, out)
    return model

# Define timesteps and n_features from previously processed data
timesteps = X_seq.shape[1]
n_features = X_seq.shape[2]

clf = build_cnn_lstm_classifier(timesteps, n_features)
clf.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
            loss='binary_crossentropy', metrics=['accuracy'])
clf.summary()

# Callbacks
callbacks = [
    EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1),
    ModelCheckpoint(os.path.join(MODEL_DIR, 'cnn_lstm_best.h5'), save_best_only=True, monitor='val_loss')
]

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 10, 78)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 10, 64)         │        15,040 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 10, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 5, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 5, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 5, 128)         │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 5, 128)         │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 2, 128)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 2, 128)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 128)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 180,417 (704.75 KB)

 Trainable params: 180,033 (703.25 KB)

 Non-trainable params: 384 (1.50 KB)

In [ ]:
# Cell 6: Train classifier
history_clf = clf.fit(X_train, y_train, validation_split=0.1, epochs=EPOCHS, batch_size=BATCH_SIZE, callbacks=callbacks)

# Save final classifier
clf.save(os.path.join(MODEL_DIR, 'cnn_lstm_final.h5'))

Epoch 1/40
538/538 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.9396 - loss: 0.1976

538/538 ━━━━━━━━━━━━━━━━━━━━ 13s 12ms/step - accuracy: 0.9396 - loss: 0.1975 - val_accuracy: 0.9664 - val_loss: 0.1063 - learning_rate: 0.0010
Epoch 2/40
538/538 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9656 - loss: 0.1088

538/538 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - accuracy: 0.9656 - loss: 0.1088 - val_accuracy: 0.9766 - val_loss: 0.0779 - learning_rate: 0.0010
Epoch 3/40
538/538 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9729 - loss: 0.0880

538/538 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.9729 - loss: 0.0880 - val_accuracy: 0.9796 - val_loss: 0.0698 - learning_rate: 0.0010
Epoch 4/40
535/538 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9758 - loss: 0.0792

538/538 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - accuracy: 0.9758 - loss: 0.0792 - val_accuracy: 0.9803 - val_loss: 0.0607 - learning_rate: 0.0010
Epoch 5/40
533/538 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9781 - loss: 0.0701

...

538/538 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.9910 - loss: 0.0237 - val_accuracy: 0.9897 - val_loss: 0.0292 - learning_rate: 5.0000e-04
Epoch 30/40
538/538 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.9920 - loss: 0.0216 - val_accuracy: 0.9904 - val_loss: 0.0301 - learning_rate: 5.0000e-04
Epoch 31/40
538/538 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.9920 - loss: 0.0223 - val_accuracy: 0.9907 - val_loss: 0.0298 - learning_rate: 5.0000e-04
Epoch 32/40
533/538 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9916 - loss: 0.0224

538/538 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - accuracy: 0.9916 - loss: 0.0224 - val_accuracy: 0.9909 - val_loss: 0.0285 - learning_rate: 5.0000e-04
Epoch 33/40
538/538 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.9920 - loss: 0.0210 - val_accuracy: 0.9907 - val_loss: 0.0293 - learning_rate: 5.0000e-04
Epoch 34/40
536/538 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9920 - loss: 0.0205

538/538 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.9920 - loss: 0.0205 - val_accuracy: 0.9917 - val_loss: 0.0283 - learning_rate: 5.0000e-04
Epoch 35/40
536/538 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9922 - loss: 0.0207

538/538 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.9923 - loss: 0.0207 - val_accuracy: 0.9917 - val_loss: 0.0277 - learning_rate: 5.0000e-04
Epoch 36/40
538/538 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.9923 - loss: 0.0203 - val_accuracy: 0.9912 - val_loss: 0.0299 - learning_rate: 5.0000e-04
Epoch 37/40
536/538 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9928 - loss: 0.0191

538/538 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.9928 - loss: 0.0191 - val_accuracy: 0.9919 - val_loss: 0.0270 - learning_rate: 5.0000e-04
Epoch 38/40
538/538 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.9925 - loss: 0.0201 - val_accuracy: 0.9907 - val_loss: 0.0304 - learning_rate: 5.0000e-04
Epoch 39/40
538/538 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - accuracy: 0.9931 - loss: 0.0200 - val_accuracy: 0.9906 - val_loss: 0.0287 - learning_rate: 5.0000e-04
Epoch 40/40
533/538 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9928 - loss: 0.0194
Epoch 40: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.
538/538 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.9928 - loss: 0.0194 - val_accuracy: 0.9916 - val_loss: 0.0287 - learning_rate: 5.0000e-04
Restoring model weights from the end of the best epoch: 37.


In [ ]:
# Cell 7: Evaluate classifier
y_pred_prob = clf.predict(X_test).ravel()
y_pred = (y_pred_prob >= 0.5).astype(int)

acc = accuracy_score(y_test, y_pred)
prec, rec, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='binary', zero_division=0)
try:
    roc_auc = roc_auc_score(y_test, y_pred_prob)
except:
    roc_auc = float('nan')

print("Classifier results: Accuracy {:.4f}, Precision {:.4f}, Recall {:.4f}, F1 {:.4f}, ROC-AUC {:.4f}".format(acc, prec, rec, f1, roc_auc))
print("\nClassification report:\n", classification_report(y_test, y_pred, target_names=['Benign','Attack']))

# Confusion matrix plot (save)
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Benign','Attack'], yticklabels=['Benign','Attack'])
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix - CNN+LSTM")
plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, "confusion_cnn_lstm.png"))
plt.close()

# ROC curve (save)
fpr, tpr, _ = roc_curve(y_test, y_pred_prob)
plt.figure(figsize=(6,4))
plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.4f}")
plt.plot([0,1],[0,1],'--')
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.title("ROC - CNN+LSTM")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, "roc_cnn_lstm.png"))
plt.close()

1194/1194 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step
Classifier results: Accuracy 0.9906, Precision 0.9431, Recall 0.8935, F1 0.9176, ROC-AUC 0.9961

Classification report:
               precision    recall  f1-score   support

      Benign       0.99      1.00      1.00     35961
      Attack       0.94      0.89      0.92      2244

    accuracy                           0.99     38205
   macro avg       0.97      0.95      0.96     38205
weighted avg       0.99      0.99      0.99     38205



In [ ]:
# Cell 8: LSTM Autoencoder for anomaly detection (zero-day)
# We'll train autoencoder on benign windows only
benign_label = le.transform(['Benign'])[0]
idx_benign_train = np.where(y_train == 0)[0]
X_ae_train = X_train[idx_benign_train]

print("Autoencoder training windows (benign):", X_ae_train.shape)

def build_lstm_autoencoder(timesteps, n_features, latent_dim=64):
    inp = Input(shape=(timesteps, n_features))
    # Encoder LSTM
    x = LSTM(128, return_sequences=True)(inp)
    x = Dropout(0.2)(x)
    x = LSTM(64, return_sequences=False)(x)
    encoded = Dense(latent_dim, activation='relu')(x)
    # Decoder
    x = RepeatVector(timesteps)(encoded)
    x = LSTM(64, return_sequences=True)(x)
    x = LSTM(128, return_sequences=True)(x)
    decoded = TimeDistributed(Dense(n_features))(x)
    model = Model(inp, decoded)
    return model

ae = build_lstm_autoencoder(timesteps, n_features, latent_dim=64)
ae.compile(optimizer='adam', loss='mse')
ae.summary()

ae_callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ModelCheckpoint(os.path.join(MODEL_DIR, 'lstm_ae_best.h5'), save_best_only=True, monitor='val_loss')
]

history_ae = ae.fit(X_ae_train, X_ae_train, validation_split=0.1, epochs=AUTOENCODER_EPOCHS, batch_size=128, callbacks=ae_callbacks)

ae.save(os.path.join(MODEL_DIR, 'lstm_ae_final.h5'))

Autoencoder training windows (benign): (143841, 10, 78)


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 10, 78)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 10, 128)        │       105,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 10, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat_vector (RepeatVector)    │ (None, 10, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 10, 64)         │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_4 (LSTM)                   │ (None, 10, 128)        │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed                │ (None, 10, 78)         │        10,062 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 301,454 (1.15 MB)

 Trainable params: 301,454 (1.15 MB)

...

1012/1012 ━━━━━━━━━━━━━━━━━━━━ 11s 11ms/step - loss: 0.3980 - val_loss: 0.4249
Epoch 5/30
1010/1012 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.3867

1012/1012 ━━━━━━━━━━━━━━━━━━━━ 11s 11ms/step - loss: 0.3867 - val_loss: 0.3997
Epoch 6/30
1012/1012 ━━━━━━━━━━━━━━━━━━━━ 11s 11ms/step - loss: 0.3763 - val_loss: 0.4578
Epoch 7/30
1010/1012 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.3760

1012/1012 ━━━━━━━━━━━━━━━━━━━━ 12s 12ms/step - loss: 0.3760 - val_loss: 0.3345
Epoch 8/30
1012/1012 ━━━━━━━━━━━━━━━━━━━━ 11s 11ms/step - loss: 0.3490 - val_loss: 0.4102
Epoch 9/30
1012/1012 ━━━━━━━━━━━━━━━━━━━━ 11s 11ms/step - loss: 0.3565 - val_loss: 0.3586
Epoch 10/30
1008/1012 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.3466

1012/1012 ━━━━━━━━━━━━━━━━━━━━ 12s 11ms/step - loss: 0.3466 - val_loss: 0.3133
Epoch 11/30
1012/1012 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.3280

1012/1012 ━━━━━━━━━━━━━━━━━━━━ 11s 11ms/step - loss: 0.3280 - val_loss: 0.2760
Epoch 12/30
1012/1012 ━━━━━━━━━━━━━━━━━━━━ 21s 11ms/step - loss: 0.3092 - val_loss: 0.3735
Epoch 13/30
1012/1012 ━━━━━━━━━━━━━━━━━━━━ 11s 11ms/step - loss: 0.3279 - val_loss: 0.5405
Epoch 14/30
1012/1012 ━━━━━━━━━━━━━━━━━━━━ 12s 11ms/step - loss: 0.3321 - val_loss: 0.3143
Epoch 15/30
1012/1012 ━━━━━━━━━━━━━━━━━━━━ 11s 11ms/step - loss: 0.3181 - val_loss: 0.3154
Epoch 16/30
1012/1012 ━━━━━━━━━━━━━━━━━━━━ 11s 11ms/step - loss: 0.3430 - val_loss: 0.3409
Epoch 16: early stopping
Restoring model weights from the end of the best epoch: 11.


In [ ]:
# Cell 9: Compute reconstruction error threshold on benign validation windows
# Use some benign windows from test set for threshold estimation
idx_benign_test = np.where(y_test == 0)[0]
X_benign_test = X_test[idx_benign_test]
recon_ben = ae.predict(X_benign_test)
mse_ben = np.mean(np.mean(np.square(recon_ben - X_benign_test), axis=2), axis=1)  # per-window MSE
thr = np.percentile(mse_ben, 95)  # threshold at 95th percentile
print("Anomaly detection MSE threshold (95th pct):", thr)

1124/1124 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step
Anomaly detection MSE threshold (95th pct): 0.6679560568619675


In [ ]:
# Cell 10: Evaluate autoencoder on test windows (both benign and attack)
recon_test = ae.predict(X_test)
mse_test = np.mean(np.mean(np.square(recon_test - X_test), axis=2), axis=1)
y_pred_ae = (mse_test > thr).astype(int)  # 1 => anomaly/attack

acc_ae = accuracy_score(y_test, y_pred_ae)
prec_ae, rec_ae, f1_ae, _ = precision_recall_fscore_support(y_test, y_pred_ae, average='binary', zero_division=0)
try:
    roc_ae = roc_auc_score(y_test, mse_test)
except:
    roc_ae = float('nan')

print("Autoencoder (anomaly) results: Accuracy {:.4f}, Precision {:.4f}, Recall {:.4f}, F1 {:.4f}, ROC-AUC {:.4f}".format(acc_ae, prec_ae, rec_ae, f1_ae, roc_ae))
print("\nClassification report (AE):\n", classification_report(y_test, y_pred_ae, target_names=['Benign','Attack']))

1194/1194 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step
Autoencoder (anomaly) results: Accuracy 0.8982, Precision 0.0789, Recall 0.0686, F1 0.0734, ROC-AUC 0.6054

Classification report (AE):
               precision    recall  f1-score   support

      Benign       0.94      0.95      0.95     35961
      Attack       0.08      0.07      0.07      2244

    accuracy                           0.90     38205
   macro avg       0.51      0.51      0.51     38205
weighted avg       0.89      0.90      0.89     38205



In [ ]:
# Save test results for Streamlit usage
np.save(os.path.join(MODEL_DIR, 'mse_test.npy'), mse_test)
np.save(os.path.join(MODEL_DIR, 'y_test.npy'), y_test)
np.save(os.path.join(MODEL_DIR, 'y_pred_clf.npy'), y_pred)
np.save(os.path.join(MODEL_DIR, 'y_pred_ae.npy'), y_pred_ae)